<a href="https://colab.research.google.com/github/bharathvaddineniK/agenticAi/blob/reactive/weather_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata
import requests

In [2]:
client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))
OPENWEATHER_KEY = userdata.get("OPENWEATHER_KEY")
GEOAPIFY_KEY = userdata.get("GEOAPIFY_KEY")

In [3]:
def get_coordinates(city_name):

    url = "https://api.geoapify.com/v1/geocode/search"

    params = {
        "text": city_name,
        "apiKey": GEOAPIFY_KEY
    }

    response = requests.get(url, params=params)
    data = response.json()

    if not data["features"]:
        return None

    coords = data["features"][0]["geometry"]["coordinates"]

    return {
        "lon": coords[0],
        "lat": coords[1]
    }

In [4]:
get_coordinates('sunnyvale')

{'lon': -122.036349, 'lat': 37.3688301}

In [5]:
def get_restaurants(city_name):

    restaurant_db = {

        "Nellore": [
            {"name": "Komala Vilas", "cuisine": "South Indian"},
            {"name": "Murali Krishna Restaurant", "cuisine": "Andhra"},
            {"name": "RRR Restaurant", "cuisine": "Indian"},
            {"name": "Seven Heaven Restaurant", "cuisine": "Multi Cuisine"}
        ],

        "San Jose": [
            {"name": "Original Joe's", "cuisine": "Italian"},
            {"name": "Orchard City Kitchen", "cuisine": "American"},
            {"name": "Back A Yard", "cuisine": "Caribbean"},
            {"name": "La Victoria Taqueria", "cuisine": "Mexican"}
        ],

        "Visakhapatnam": [
            {"name": "Dharani Restaurant", "cuisine": "Andhra"},
            {"name": "Vista Restaurant", "cuisine": "Seafood"},
            {"name": "The Square", "cuisine": "Multi Cuisine"},
            {"name": "Upland Bistro", "cuisine": "Continental"}
        ],

        "Kurnool": [
            {"name": "Hotel Mourya Inn", "cuisine": "Indian"},
            {"name": "Abhiruchi Restaurant", "cuisine": "South Indian"},
            {"name": "Sri Krishna Grand", "cuisine": "Multi Cuisine"},
            {"name": "Sai Sagar Restaurant", "cuisine": "Vegetarian"}
        ],

        "Sunnyvale": [
            {"name": "DishDash", "cuisine": "Middle Eastern"},
            {"name": "Falafel STOP", "cuisine": "Mediterranean"},
            {"name": "Rok Bistro", "cuisine": "Asian Fusion"},
            {"name": "La Fiesta", "cuisine": "Mexican"}
        ]

    }

    return restaurant_db.get(city_name, [])

In [6]:
def get_activities(city_name):

    activity_db = {

        "Nellore": [
            {"name": "Mypadu Beach Visit", "type": "outdoor"},
            {"name": "Nellapattu Bird Sanctuary", "type": "outdoor"},
            {"name": "Ranganatha Temple Visit", "type": "indoor"},
            {"name": "VR Mall Shopping", "type": "indoor"}
        ],

        "San Jose": [
            {"name": "Santana Row Shopping", "type": "outdoor"},
            {"name": "Municipal Rose Garden Walk", "type": "outdoor"},
            {"name": "Winchester Mystery House Tour", "type": "indoor"},
            {"name": "Tech Interactive Museum", "type": "indoor"}
        ],

        "Visakhapatnam": [
            {"name": "RK Beach Walk", "type": "outdoor"},
            {"name": "Kailasagiri Hill Park", "type": "outdoor"},
            {"name": "INS Kursura Submarine Museum", "type": "indoor"},
            {"name": "Visakhapatnam Zoo Visit", "type": "outdoor"}
        ],

        "Kurnool": [
            {"name": "Konda Reddy Fort Visit", "type": "outdoor"},
            {"name": "Oravakallu Rock Garden", "type": "outdoor"},
            {"name": "Belum Caves Exploration", "type": "indoor"},
            {"name": "Kurnool Mall Shopping", "type": "indoor"}
        ],

        "Sunnyvale": [
            {"name": "Baylands Park Walk", "type": "outdoor"},
            {"name": "Stevens Creek Trail Cycling", "type": "outdoor"},
            {"name": "Sunnyvale AMC Theater", "type": "indoor"},
            {"name": "Computer History Museum Visit", "type": "indoor"}
        ]

    }

    return activity_db.get(city_name, [])

In [7]:
def get_city_weather(city_name: str) -> str:
  print(f'getting the weather for {city_name}...')
  weather = {
      'nellore': {"temp": 65, "condition": "cloudy"},
      'san jose': {"temp": 75, "condition": "hot"},
      'visakhapatnam': {"temp": 85, "condition": "very hot"},
      'kurnool': {"temp": 95, "condition": "super hot"},
      'sunnyvale': {"temp": 105, "condition": "very very hot"}
  }
  result = weather.get(city_name.lower(), {'error': 'Invalid city name'})
  return json.dumps(result)

In [8]:
AVAILABLE_FUNCTIONS = {
    'get_city_weather': get_city_weather,
    'get_restaurants': get_restaurants,
    'get_activities': get_activities
}

In [9]:
tools_schema = [

    {
        "type": "function",
        "function": {
            "name": "get_city_weather",
            "description": "Get the weather based on the given city name",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {
                        "type": "string",
                        "description": "The name of a city"
                    }
                },
                "required": ["city_name"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_restaurants",
            "description": "Get a list of restaurants available in a given city including the cuisine they serve",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {
                        "type": "string",
                        "description": "The name of the city"
                    }
                },
                "required": ["city_name"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_activities",
            "description": "Get a list of activities available in a city. Activities may be indoor or outdoor.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {
                        "type": "string",
                        "description": "The name of the city"
                    }
                },
                "required": ["city_name"]
            }
        }
    }

]

In [12]:
def run_it_agent(userPrompt: str):
  print(f"\n getting weather for a new place....")
  messages = [
      {"role":"user", "content":userPrompt},
      {
 "role": "system",
 "content": """
You are a city planning assistant.

You help users understand the weather in a city and suggest suitable activities and restaurants.

Available tools:
- get_city_weather
- get_restaurants
- get_activities

Workflow:

1. If the user asks about a city, first retrieve the weather using get_city_weather.
2. Based on the weather, decide whether outdoor or indoor activities are appropriate.

Weather reasoning:
- Sunny / pleasant → outdoor activities are preferred.
- Cold / rainy / bad weather → indoor activities are preferred.

3. Retrieve activities using get_activities.
4. Select only activities that match the weather conditions.
5. Retrieve restaurants using get_restaurants if food or evening plans are relevant.
6. Recommend 1–2 good activities and optionally 1 restaurant.

Important:
- Do NOT list all activities or restaurants returned by tools.
- Choose only a few appropriate recommendations.
- Create a short plan rather than dumping tool results.
- Do not ask any followup questions.

Response structure:

Weather summary
Recommended activity
Optional restaurant suggestion
"""
}
  ]
  while True:
      print("\n AI thinking...")
      response = client.chat.completions.create(
          model = 'gpt-4o',
          messages = messages,
          tools = tools_schema,
          tool_choice = 'auto'
      )
      response_msg = response.choices[0].message
      messages.append(response_msg)
      if response_msg.tool_calls:
        for tool_call in response_msg.tool_calls:
          func_name = tool_call.function.name
          func_args = json.loads(tool_call.function.arguments)
          function_to_call = AVAILABLE_FUNCTIONS.get(func_name)
          if function_to_call:
              function_response = function_to_call(**func_args)
              messages.append({
                  "role":"tool",
                  "content": json.dumps(function_response),
                  "name": func_name,
                  "tool_call_id": tool_call.id
              })
      else:
        print(f'\n Final response: {response_msg.content}')
        break


In [13]:
run_it_agent('I am thinking what should i do at this point of day in nellore')


 getting weather for a new place....

 AI thinking...
getting the weather for nellore...

 AI thinking...

 Final response: Weather in Nellore is cloudy with a temperature of 65°F. 

Recommended Activity: As there are no specific activities listed, I suggest exploring local city sights or enjoying a cozy time at a local café or mall.

Optional Restaurant Suggestion: If you're interested in a meal, I can find some restaurants for you. Would you like that?


In [15]:
run_it_agent("How's sunnyvale doing")


 getting weather for a new place....

 AI thinking...
getting the weather for Sunnyvale...

 AI thinking...

 Final response: Weather Summary: It's very hot in Sunnyvale with a temperature around 105°F.

Recommended Activity: Given the extreme heat, it's better to opt for indoor activities. You can visit the Computer History Museum for an engaging and cool experience.

Optional Restaurant Suggestion: If you're interested in dining options, let me know, and I can find a suitable restaurant for you!
